##### Process Addresses Data
1. Ingest the data into the data lakehouse - bronze_address
2. Perform data quality checks and transform the data as required - silver_address_clean
3. Apply changes to the Addresses data (SCD Type 2) - siler_addresses


In [0]:
import dlt
import pyspark.sql.functions as F



##### 1. Ingest the data into the data lakehouse - bronze_address

In [0]:
@dlt.table(
    name ="bronze_address",
    table_properties = {'quality': 'bronze'},
    comment = "Raw  addresses data ingested from the source system",
)
def create_bronze_address():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.inferColumnTypes", "true")
        .load("/Volumes/circuitbox/landing/operations_data/addresses/")
        
        .select("*",
                F.col("_metadata.file_path").alias("input_file_path"),
                F.current_timestamp().alias("ingest_timestamp")
        )
)


##### 2. Perform data quality checks and transform the data as required - silver_addresses_clean

Silver_Address_clean (DQ Rules)
1. Fail if customer_id is null
2. Drop records with address_line_1 as null
3. warn if postcode is not 5 digits

Sivlver_Address_Clean (Transofrmations)
1. CAST created_date to DATE

Silver_Addresses is a Type 2 Dimension Table with primary key as Customer_id. Appy changes to the addresses data based on created_date and create history for each address added for the customer.

In [0]:
@dlt.table(
    name ="silver_addresses_cleaned",
    table_properties = {'quality': 'silver'},
    comment = "Raw  addresses data ingested from the source system"
)
@dlt.expect_or_fail("valid_customer_id", "customer_id is not null")
@dlt.expect_or_drop("valid_address", "address_Line_1 is not null")
@dlt.expect("valid_postcode","LENGTH(postcode) = 5")
def create_silver_addresses_cleaned():
    return (
        spark.readStream.table("Live.bronze_address")
        .select("customer_id",
                "address_line_1",
                "city",
                "state",
                "postcode",
                F.col("created_date").cast("date").alias("created_date"),
                F.col("_metadata.file_path").alias("input_file_path")
        )      

    )


##### 3. Apply changes to the Addresses date (SCD Type 2) - silver_adderess

In [0]:
dlt.create_streaming_table(
    name = "silver_addresses",
    comment = "SCD type 2 dimension addresses data",
    table_properties = {'quality': 'silver'}    
)
dlt.apply_changes(
    target = "silver_addresses",
    source = "LIVE.silver_addresses_cleaned",
    keys = ["customer_id"],
    sequence_by = F.col("created_date"),
    stored_as_scd_type = 2
)